# Creating a multimodal bot 

### this bot takes audio or text as input and response in markdown and audio

### it also have tool use and ability to create Images and choose between gpt or claude models

# this bot is a tech assistant bot

it name is mamad it have a very long system prompt and the ability to use the web_search tool to get the most recent info about the topic the web_search tool is just a basic tavily api call
and it have the current date and time in its system prompt so it don't say outdated things

# how I handle audio processing

1- user speaks with audio

2- with whisper I transcribe the audio into text 

3- give that text to an LLM

4- LLM generates response

5- with gTTS transform that text into audio and give the text and the audio to user

# Important

### remeber to download the ffmpeg first otherwise you might get some errors 

here are the links:

https://ffmpeg.org/download.html

https://www.gyan.dev/ffmpeg/builds


also install whisper and gtts first with this command 

`pip install openai-whisper gtts` 

or 

`uv add openai-whisper gtts` 

make sure that the virtual environment is active in the llm-engineering directory.

you should be seeing (llm-engineering) before the PS ....... if you don't see that in the 
 
llm-engineering directory type this command in windows:

` .venv\Scripts\activate`

or this in mac or linux:

`source .venv/bin/activate`

in windows if you are seeing some sort of execution policy error probably your Execution policy is changed to something that don't allow running scripts check this website for solution:

https://learn.microsoft.com/en-us/powershell/module/microsoft.powershell.core/about/about_execution_policies?view=powershell-7.6



## so enough of talking lets start building

#### we start with some imports

In [ ]:
import json
import os 
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from tavily import TavilyClient
from datetime import datetime
import whisper
from gtts import gTTS
import tempfile
import base64
from io import BytesIO
import PIL.Image as Image

## initializing api keys 

note that I am in a country named Iran and I don't have access to dollors so I can't get openai API keys but I am using a different provider in my country named avalai that is like openrouter but it accepts Iran's currency make sure to change this to the provider that you choose like openai or ollama if you are doing it local 

also make sure that you have a tavily account and an API key set, here is the link to tavily:

https://tavily.com/

In [ ]:
load_dotenv(override=True)

avalai_api_key = os.getenv("AVALAI_API_KEY") # change this to your provider
tavily_api_key = os.getenv("TAVILY_API_KEY")

if avalai_api_key:
    print(f"avalai api key exsits and starts with {avalai_api_key[:5]}")

else:
    print("avalai api key does not exsits")

if tavily_api_key:
    print(f"tavily api key exsits and starts with {tavily_api_key[:7]}")

else:
    print("tavily api key is not set")
    
avalai_url = "https://api.avalai.ir/v1"

avalai = OpenAI(base_url=avalai_url, api_key=avalai_api_key)

gpt_model = "gpt-4.1-mini"
claude_model = "claude-haiku-4-5"

## loading the whisper model

#### if this is your first time it might take some times be patient.

In [ ]:
whisper_model = whisper.load_model("base")

### now we create that web_search tool function with its json

In [ ]:
def web_search(query):
    tavily_client = TavilyClient(api_key=tavily_api_key)
    response = tavily_client.search(query)
    print(f"tool called with query: {query}")
    content = []
    for i in response["results"]:
        content.append({"url":i["url"], "title":i["title"], "content":i["content"]})
    return content


In [ ]:
web_search_json = {
    "name":"web_search",
    "description":"always use this tool to get the most recent information about the topic",
    "parameters":{
        "type":"object",
        "properties":{
            "query":{
                "type":"string",
                "description":"the questions that the user have in a short 3 to 4 words query"
            },
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [{"type":"function", "function":web_search_json}]

# now it is time for system prompts

there are 3 system prompts:

1- the bot mamad system prompt that have the current date and time in it

2- a system prompt that tells an LLM to generate prompts for an Image generation model base on the answer that the bot provides to make a good illustration 

3- the last one is one that tells an LLM to see if an answer is good to make a picture of that answer basically not every message like hello, greeting, etc.. are not important what we do is that we use an LLM to see if an answer is actually important or not we use pydantic to get structured output

#### if you fell confused read the prompts they are so big I actually tell chatgpt to inhance them don't think I wrote all of them

In [ ]:
system_prompt = f"""
You are an expert AI technical assistant named Mamad with over 20 years of experience in programming, software engineering, Python, APIs, AI engineering, Git, backend systems, and developer tools.

Your job is to explain technical concepts in the simplest and most beginner-friendly way possible so users never feel confused or overwhelmed.

You specialize in Python programming, but you can also answer questions about:

* JavaScript
* TypeScript
* Docker
* APIs
* Git and GitHub
* AI agents
* LLMs
* databases
* web development
* cloud tools
* software engineering concepts

Always:

* sound helpful, friendly, polite, and slightly funny
* explain concepts using simple real-world examples
* teach step-by-step
* avoid overly academic explanations
* respond only in markdown
* use headings, bullet points, and code blocks when useful
* never hallucinate information
* if uncertain, clearly say you are unsure

You have access to a tool called:
web_search(query)

VERY IMPORTANT TOOL RULES:

* ALWAYS use the web_search tool before answering any technical question
* ALWAYS search for the latest information, documentation, versions, APIs, or updates
* NEVER answer purely from memory for technical topics
* Break the user's question into a short 3-to-4-word search query before using web_search
* If needed, use multiple short queries

Examples of good search queries:

* "latest python asyncio"
* "docker compose v2"
* "langgraph memory system"
* "openai responses api"

After receiving web results:

* summarize the information clearly
* explain it in beginner-friendly language
* include practical examples when possible
* avoid copying documentation directly

One-shot example:

User:
"What is the difference between Git and GitHub?"

Assistant thinking process:

1. Convert question into short search query:
   "git github difference"
2. Use:
   web_search("git github difference")
3. Read latest information
4. Answer simply

Example response style:

# Git vs GitHub

Think of Git like a save system for your code.

It tracks changes in your project just like a video game save file.

GitHub is like Google Drive for Git projects — it lets developers store and share their code online.

## Simple Real-World Example

* Git = Photoshop save history on your computer
* GitHub = Cloud storage where you upload and collaborate with others

## Main Difference

| Git                       | GitHub                         |
| ------------------------- | ------------------------------ |
| Tool installed on your PC | Website/platform               |
| Tracks code changes       | Stores Git repositories online |
| Works offline             | Mostly online                  |
| Made for version control  | Made for collaboration         |

Always keep explanations easy to understand even for beginners.

todays date and time is {datetime.now()} this is the correct date and time
the latest news data is this time never use the date and time that you think is the latest

Important note:

cause your answers are being transformed into audio, please avoid using emojis, special characters, or complex markdown that may not convert well to speech. Focus on clear and concise explanations in plain text with simple formatting when necessary.
"""

system_image_prompt = f"""
You are an expert AI prompt engineer specialized in converting chatbot responses into high-quality image generation prompts.

The chatbot's name is Mamad, and its responses are written in markdown format.

Your task is to:

1. Read and fully understand the chatbot response.
2. Extract the most visually important ideas, concepts, objects, scenes, emotions, and environments.
3. Ignore unnecessary markdown syntax, code blocks, links, or formatting.
4. Convert the response into a single detailed cinematic image generation prompt.
5. Create prompts that are visually rich, descriptive, and optimized for modern AI image generation models.

Image Style Requirements:
- 3D animated style
- colorful and vibrant
- cinematic lighting
- highly detailed
- visually attractive
- modern animated movie aesthetic
- expressive characters and environments
- clean composition
- professional quality
- dynamic atmosphere

Prompt Writing Rules:
- Focus on visual storytelling.
- Describe the subject, environment, lighting, mood, camera angle, colors, and important objects.
- Keep the prompt concise but detailed.
- Do NOT explain the image.
- Do NOT summarize the chatbot response.
- Output ONLY the final image generation prompt.
- Avoid markdown formatting in the output.
- Avoid phrases like "generate an image of".
- Make the final result feel like a scene from a high-budget animated movie.

Example:

Chatbot Response:
"Transformers are AI models that understand relationships between words using attention mechanisms."

Output:
A futuristic 3D animated AI laboratory with giant glowing neural networks connecting floating words and symbols in the air, colorful holographic attention lines linking data nodes, cinematic blue and purple lighting, highly detailed sci-fi environment, modern Pixar-style 3D animation, ultra detailed, vibrant atmosphere, dynamic camera angle, professional animated movie quality
"""

system_important_prompt = """
You are an expert AI classifier specialized in analyzing chatbot responses and deciding whether they should be converted into an image.

The chatbot answers technical and educational questions related to programming, AI, software engineering, APIs, Docker, machine learning, and other technology topics.

Your task is to determine if the chatbot response contains visually meaningful educational content that would benefit from image generation.

Return:
- True → if the response explains an important concept, workflow, architecture, comparison, tutorial, system, process, or contains rich educational/technical information that could be visualized.
- False → if the response is conversational, short, generic, emotional, or does not contain meaningful visual/educational information.

Important Rules:
- Messages like greetings, thanks, confirmations, jokes, or small talk are NOT important.
- Very short answers are usually NOT important.
- Detailed explanations ARE important.
- Step-by-step tutorials ARE important.
- Technical architecture explanations ARE important.
- Comparisons between technologies ARE important.
- Responses containing rich concepts, systems, diagrams, workflows, or educational value ARE important.
- Ignore markdown formatting when making the decision.
- Be conservative: only return True when the response would genuinely benefit from a generated image.

Output Rules:
- Output ONLY:
True

OR

False

- Do NOT explain your reasoning.
- Do NOT output markdown.
- Do NOT output extra text.
- Do NOT output punctuation.

Examples:

Input:
"Hello! How can I help you today?"

Output:
False

Input:
"Transformers use attention mechanisms to understand relationships between words in a sentence."

Output:
True

Input:
"Thanks! Glad I could help."

Output:
False

Input:
"Docker containers package applications with their dependencies so they can run consistently across environments."

Output:
True

Input:
"Sure, I can help with that."

Output:
False
"""

## now making that handle tool call function that handles tool calls

In [ ]:
def handle_tool_call(message):
    result = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        res = tool(**arguments) if tool else {}
        result.append({"role": "tool", "content": json.dumps(res), "tool_call_id": tool_call.id})

    return result

## now we use pydantic to force the model to generate the output we want

#### in this case we only want to know is it important or not so it is a boolean(True or False)

In [ ]:
from pydantic import BaseModel 

class is_important_response(BaseModel): # create a subclass of basemodel
    decision: bool

### now we write three functions

1- one that generates the prompt

2- one that creates the picture

3- one that see if the message is important or not if it is, it runs the create picture function if not it returns not important


In [ ]:
def generate_prompt(bot_text):
    messages = [{"role": "system", "content": system_image_prompt}, 
                {"role": "user", "content": f"here is the chatbot response:\n{bot_text}"}]
    response = avalai.chat.completions.create(model="gemini-2.5-flash", messages=messages)
    return response.choices[0].message.content

In [ ]:
def create_picture(bot_text):
    prompt = generate_prompt(bot_text)
    response = avalai.images.generate(model="gpt-image-1", 
                                      prompt=prompt, 
                                      size="1024x1024",
                                        n=1,
                                        response_format="b64_json")
    
    image_data_b64 = response.data[0].b64_json
    image_data = base64.b64decode(image_data_b64)
    image = Image.open(BytesIO(image_data))
    return image


In [ ]:
def see_important(bot_text):
    messages = [{"role": "system", "content": system_important_prompt}, 
                {"role": "user", "content": f"here is the chatbot response:\n{bot_text}"}]
    
    # this response format is is_important_response so it returns a json
    
    response = avalai.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=is_important_response)
    decide = json.loads(response.choices[0].message.content)
    print(decide["decision"])
    if decide["decision"] == True:
        return create_picture(bot_text)
    else:
        return "not important"

## now we write two other functions

1- one that transform speech to text (stt)

2- one that transform text to speech (tts)

In [ ]:
def stt(audio):
    result = whisper_model.transcribe(audio) # transcribe that audio to text
    return result["text"]

In [ ]:
def tts(text):
    tts = gTTS(text)
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    tts.save(temp_file.name)
    return temp_file.name

## now making the chat function

##### this chat function takes 5 inputs:

1- audio

2- text

3- the model that the user choose, it is a gr.Dropdown you can add more and more models if you wish so

4- pic_want, this is also dropdown and it is yes or no if the user choose yes it generates pictures for important answers and if no it never generates any pictures

5- history, the thing that give this bot memory (the whole conversation)

In [ ]:
def chat(audio, text, model, pic_want, history):

    if history is None:
        history = []

    if audio is not None:
        result = whisper_model.transcribe(audio)
        user_text = result["text"]

    elif text.strip():
        user_text = text    

    else:
        return None, history, None, history
    
    print(f"user said: {user_text}")

    history.append({"role":"user", "content":user_text})

    messages = [{"role":"system", "content":system_prompt}] 

    messages.extend(history)
    
    select_model=( # choosing the model
        gpt_model
        if model.upper == "GPT"
        else claude_model
    )

    response = avalai.chat.completions.create(model=select_model, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        res = handle_tool_call(message)
        messages.append(message)
        messages.extend(res)
        response = avalai.chat.completions.create(model=select_model, messages=messages, tools=tools)
        
    bot_text = response.choices[0].message.content

    print(f"assistant say: {bot_text}")

    history.append({"role":"assistant", "content":bot_text})

    if pic_want.upper() == "YES":
        pic = see_important(bot_text)

    elif pic_want.upper() == 'NO':
        pic = "don't want pic"

    else:
        raise ValueError("you can only say yes or no")

    return tts(bot_text), bot_text, pic, history


# finally the moment of truth

## making gradio blocks and launching it

### the `gr.State` give the model memory 

In [ ]:
with gr.Blocks() as ui:

    state = gr.State([])
    gr.Markdown("Mamad AI Voice and chat Tech assistant")

    with gr.Row():
        markdown = gr.Markdown(height=500) 
        
        image_output = gr.Image(height=500, interactive=False)
    
    with gr.Row():
        audio_input = gr.Audio(label="speak here",
                               sources=["microphone"],
                               type="filepath")
        
        text_input = gr.Textbox(label="or type here")

    with gr.Row():
        audio_output = gr.Audio(label="Your answer",
                                autoplay=True)
        
        model_choice = gr.Dropdown(label="choose a model", choices=["GPT", "Claude"], value="GPT")

        pic_want = gr.Dropdown(label="Want image for better understanding (it will only work for tech answers not general things like Hello and it may take some time)", choices=["Yes", "No"], value="No")

    submit_btn = gr.Button("send")

    submit_btn.click(fn=chat,      # this is the thing that tells the gradio in which order put this components together
                     inputs=[audio_input, text_input, model_choice, pic_want,
                              state],
                     outputs=[audio_output, markdown, image_output, state])
    

ui.launch() # yeahhhhhhhhhhhh


### if you don't understand any part of the code:

1- add in print statements

2- change it 

3- ask AI to explain the code for you

## my recomendation is to make your own multimodal bot that actually solve a real problem for you